# 2.1 监督学习 (Supervised Learning)

从带有标签的数据中学习输入到输出的映射关系。

本节涵盖：
- 全监督学习
- 监督微调（SFT）
- 奖励模型训练
- 多标签监督

> 🕐 预估学习时间：35分钟


## 1. 全监督学习

**目的**：从完整标注数据中学习精确的任务映射

**基本原理**：给定输入-标签对{(x_i, y_i)}，通过最小化预测值与真实标签之间的损失函数来优化模型参数。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

class SimpleTextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        embeds = self.embedding(x).mean(dim=1)
        hidden = self.relu(self.fc1(embeds))
        return self.fc2(hidden)

vocab_size, embed_dim, hidden_dim, num_classes = 1000, 64, 32, 3
model = SimpleTextClassifier(vocab_size, embed_dim, hidden_dim, num_classes)

n_samples = 500
seq_len = 20
X = torch.randint(0, vocab_size, (n_samples, seq_len))
y = torch.randint(0, num_classes, (n_samples,))

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print('=== Full Supervised Learning Demo ===')
for epoch in range(10):
    total_loss = 0
    correct = 0
    total = 0
    for batch_X, batch_y in loader:
        optimizer.zero_grad()
        logits = model(batch_X)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (logits.argmax(dim=-1) == batch_y).sum().item()
        total += batch_y.size(0)
    if (epoch + 1) % 2 == 0:
        print(f'Epoch {epoch+1}: loss={total_loss/len(loader):.4f}, acc={correct/total:.4f}')

print(f'\nFinal accuracy: {correct/total:.4f}')

## 2. 监督微调（SFT）

**目的**：使预训练模型适配特定任务格式

**基本原理**：在预训练模型上使用指令-回复对进行监督训练，损失通常仅计算回复部分。

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print('=== SFT Loss Computation Demo ===')

tokenizer = AutoTokenizer.from_pretrained('gpt2')
model = AutoModelForCausalLM.from_pretrained('gpt2')
model.eval()

prompt = "Translate to French: Hello"
response = " Bonjour"
full_text = prompt + response

full_ids = tokenizer.encode(full_text, return_tensors='pt')
prompt_len = len(tokenizer.encode(prompt))

with torch.no_grad():
    outputs = model(full_ids, labels=full_ids)
    full_loss = outputs.loss

# SFT: only compute loss on response tokens
response_ids = full_ids[:, prompt_len:]
with torch.no_grad():
    resp_outputs = model(full_ids)
    logits = resp_outputs.logits[:, prompt_len-1:-1, :]
    loss_fct = torch.nn.CrossEntropyLoss()
    sft_loss = loss_fct(logits.reshape(-1, logits.size(-1)), response_ids.reshape(-1))

print(f'Prompt: "{prompt}"')
print(f'Response: "{response}"')
print(f'Full text loss: {full_loss.item():.4f}')
print(f'SFT loss (response only): {sft_loss.item():.4f}')
print(f'\nKey insight: SFT only backpropagates through response tokens,')
print(f'not the prompt tokens. This teaches the model to generate correct responses.')

## 3. 奖励模型训练

**目的**：从人类偏好数据中学习评分函数

**基本原理**：将人类偏好排序转化为二元偏好对，训练奖励模型使偏好回复的得分高于非偏好回复。

In [ ]:
import torch
import torch.nn as nn

class RewardModel(nn.Module):
    def __init__(self, hidden_dim=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(64, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, x):
        return self.encoder(x).squeeze(-1)

def bradley_terry_loss(chosen_rewards, rejected_rewards):
    """Bradley-Terry preference model loss for reward model training.
    
    L = -log(sigma(r_chosen - r_rejected))
    """
    return -torch.log(torch.sigmoid(chosen_rewards - rejected_rewards)).mean()

torch.manual_seed(42)
reward_model = RewardModel()
optimizer = torch.optim.Adam(reward_model.parameters(), lr=1e-3)

n_pairs = 200
chosen_features = torch.randn(n_pairs, 64) + 0.5
rejected_features = torch.randn(n_pairs, 64) - 0.5

print('=== Reward Model Training (Bradley-Terry) ===')
for epoch in range(20):
    chosen_r = reward_model(chosen_features)
    rejected_r = reward_model(rejected_features)
    loss = bradley_terry_loss(chosen_r, rejected_r)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    accuracy = (chosen_r > rejected_r).float().mean()
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1}: loss={loss.item():.4f}, preference_accuracy={accuracy.item():.4f}')

with torch.no_grad():
    test_chosen = reward_model(chosen_features[:10])
    test_rejected = reward_model(rejected_features[:10])
    print(f'\nSample rewards: chosen_mean={test_chosen.mean():.3f}, rejected_mean={test_rejected.mean():.3f}')
    print(f'Chosen > Rejected: {(test_chosen > test_rejected).sum().item()}/10')

## 4. 多标签监督

**目的**：处理一个输入对应多个标签的场景

**基本原理**：使用多个二分类头或sigmoid损失，适用于安全分类、主题标注等多标签任务。

In [ ]:
import torch
import torch.nn as nn

class MultiLabelClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_labels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_labels),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.net(x)

torch.manual_seed(42)
input_dim, hidden_dim, num_labels = 64, 32, 5
label_names = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult']

model = MultiLabelClassifier(input_dim, hidden_dim, num_labels)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_samples = 500
X = torch.randn(n_samples, input_dim)
y = torch.zeros(n_samples, num_labels)
for i in range(n_samples):
    for j in range(num_labels):
        y[i, j] = 1.0 if X[i, 0] * (j + 1) * 0.3 + torch.randn(1).item() * 0.5 > 0.5 else 0.0

print('=== Multi-Label Supervised Learning ===')
for epoch in range(20):
    preds = model(X)
    loss = criterion(preds, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 5 == 0:
        predicted = (preds > 0.5).float()
        per_label_acc = (predicted == y).float().mean(dim=0)
        print(f'Epoch {epoch+1}: loss={loss.item():.4f}')
        for j, name in enumerate(label_names):
            print(f'  {name:15s}: acc={per_label_acc[j]:.3f}, pos_rate={y[:, j].mean():.3f}')

print('\nMulti-label vs single-label: each label is independently predicted with sigmoid + BCE loss.')

## 📝 课后思考题1. **全监督学习与监督微调（SFT）在大模型场景下的主要区别是什么？**   <details><summary>💡 点击查看提示</summary>      **参考答案**：   - **全监督学习**：从零开始训练模型，使用完整标注数据学习输入到输出的映射关系。在大模型场景下，通常用于特定任务的分类、回归等，数据量需求大。   - **监督微调（SFT）**：在预训练模型基础上，使用指令-回复对进行监督训练。关键区别在于：(1) 基于已有知识迁移，而非从零学习；(2) 损失通常仅计算回复部分（Response），不计算Prompt部分的损失；(3) 目的是让模型学会遵循指令格式，而非学习基础的语言理解能力。   - **本质区别**：全监督学习学习任务本身，SFT学习任务格式和交互模式。      </details>2. **奖励模型训练中，如何设计偏好数据收集策略以减少标注者偏见？**   <details><summary>💡 点击查看提示</summary>      **参考答案**：   - **多标注者交叉**：每条数据由多个标注者独立评分，使用多数投票或Bradley-Terry扩展到多标注者场景。   - **去偏差化指令**：制定详细、客观的标注指南，减少主观性；例如明确"正确性 > 完整性 > 流畅性"的优先级。   - **对抗性采样**：在偏好对中混入明显优劣的对照组，检测标注者是否认真。   - **位置偏差消除**：随机交换chosen和rejected的回答顺序，因为标注者往往偏好第一个看到的回答。   - **长度偏差控制**：使用长度归一化的奖励或刻意包含短优/长劣的对照样本。   - **高质量反馈来源**：结合人类标注和AI反馈（RLAIF），利用强模型进行辅助评分。      </details>3. **分类任务和回归任务在损失函数选择上有什么不同的考量？**   <details><summary>💡 点击查看提示</summary>      **参考答案**：   - **分类任务**：     - 二分类：BCE Loss（二元交叉熵），适用于sigmoid输出。     - 多分类：CrossEntropyLoss（交叉熵），内置softmax，类别互斥。     - 多标签分类：BCE Loss（每个标签独立sigmoid），标签不互斥。     - 类别不平衡：Focal Loss、加权交叉熵、Dice Loss。   - **回归任务**：     - MSE Loss：对异常值敏感，梯度与误差成正比，适合数据干净场景。     - MAE Loss：对异常值鲁棒，梯度恒定，但零点不可导（需smooth L1）。     - Huber Loss：结合MSE和MAE，小误差用MSE（平滑），大误差用MAE（鲁棒）。   - **核心考量**：分类关注概率分布对齐（交叉熵族），回归关注距离度量选择（L1/L2族）；异常值敏感性、梯度特性、任务数值范围都是关键因素。      </details>